## 量化FP32模型并导出单个测试图片计算过程中的所有INT8数据
1. **其中Scale参数来源于训练集采样** 
2. 导出原始参数存放在test_vector.h中

In [22]:
import torch
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import numpy as np

# ==========================================
# 0. 定义手动展平函数
# ==========================================
def my_flatten(x):
    """
    手动实现展平操作，严格按照 C/C++ 的行优先 (Row-Major) 逻辑。
    """
    B, C, H, W = x.shape
    flattened_memory = []
    for b in range(B):
        for c in range(C):
            for h in range(H):
                for w in range(W):
                    val = x[b, c, h, w].item() 
                    flattened_memory.append(val)
    return torch.tensor([flattened_memory], dtype=x.dtype, device=x.device)

# ==========================================
# 1. 融合 BN 后的纯净全网定义 (Hardware-Ready)
# ==========================================
class HardwareFriendlyCNNFused(torch.nn.Module):
    def __init__(self):
        super(HardwareFriendlyCNNFused, self).__init__()
        # 直接读取融合后的通道数配置: 3->64->64->64->64
        self.conv1 = torch.nn.Conv2d(3, 64, kernel_size=3, padding=1)
        self.conv2 = torch.nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.pool1 = torch.nn.MaxPool2d(2, 2)
        
        self.conv3 = torch.nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.conv4 = torch.nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.pool2 = torch.nn.MaxPool2d(2, 2)
        
        self.fc = torch.nn.Linear(64 * 8 * 8, 10)
        self.relu = torch.nn.ReLU()
        # 推理时不需要 dropout，这里删去
        
    def forward(self, x):
        # 严格对齐 train_cnn.py 里的前向计算逻辑
        x = self.relu(self.conv1(x))
        x = self.pool1(self.relu(self.conv2(x)))
        x = self.relu(self.conv3(x))
        x = self.pool2(self.relu(self.conv4(x)))
        x = x.view(x.size(0), -1) 
        x = self.fc(x)
        return x

# ==========================================
# 2. 量化核心工具函数
# ==========================================
def sym_quantize(tensor):
    """对称量化：强制 Zero Point = 0"""
    max_abs = tensor.abs().max().item()
    scale = max_abs / 127.0 if max_abs > 0 else 1.0
    q_tensor = torch.round(tensor / scale)
    q_tensor = torch.clamp(q_tensor, -127, 127).to(torch.int8)
    return q_tensor, scale

def export_c_array(tensor, name, file_handle, c_type="int8_t"):
    """导出为一维 C 数组"""
    flat_data = tensor.flatten().tolist()
    file_handle.write(f"const {c_type} {name}[{len(flat_data)}] = {{\n    ")
    for i, val in enumerate(flat_data):
        file_handle.write(f"{val}, ")
        if (i + 1) % 16 == 0:
            file_handle.write("\n    ")
    file_handle.write("\n};\n\n")

def calc_multiplier_shift(s_in, s_w, s_out, shift=16):
    """计算硬件所需的 Multiplier，Shift 固定为 16"""
    M = (s_in * s_w) / s_out
    multiplier = int(round(M * (1 << shift)))
    return multiplier, shift

# ==========================================
# 3. 主流程：计算 Scale，生成参考模型
# ==========================================
def main():
    print(">>> 步骤 1/4: 加载融合版模型与数据集...")
    model = HardwareFriendlyCNNFused()
    # 读取上一步生成的无 BN 纯权重！
    model.load_state_dict(torch.load('my_hardware_cnn_fp32_64ch_fused.pth', weights_only=True))
    model.eval()
    
    transform = transforms.Compose([transforms.ToTensor()])
    
    # 训练集用于校准 PTQ 的极值
    trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
    calib_loader = torch.utils.data.DataLoader(torch.utils.data.Subset(trainset, range(1000)), batch_size=100, shuffle=False)
    
    # 测试集用于提取 Index=1 作为验证样本
    testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
    
    TARGET_INDEX = 1
    target_img_fp32, target_label = testset[TARGET_INDEX]
    target_img_fp32 = target_img_fp32.unsqueeze(0) 
    print(f"提取黄金测试图片成功 (Index={TARGET_INDEX})！真实标签: {target_label}")

    print("\n>>> 步骤 2/4: 执行严格的 PTQ 静态校准 (基于训练集)...")
    global_max_in = 0.0
    global_max_out1, global_max_out2, global_max_out3, global_max_out4 = 0.0, 0.0, 0.0, 0.0
    global_max_fc = 0.0

    with torch.no_grad():
        for images, _ in calib_loader:
            global_max_in = max(global_max_in, images.abs().max().item())
            
            out1 = F.conv2d(images, model.conv1.weight, model.conv1.bias, padding=1)
            global_max_out1 = max(global_max_out1, out1.abs().max().item())
            act1 = F.relu(out1)
            
            out2 = F.conv2d(act1, model.conv2.weight, model.conv2.bias, padding=1)
            global_max_out2 = max(global_max_out2, out2.abs().max().item())
            act2 = F.max_pool2d(F.relu(out2), 2)
            
            out3 = F.conv2d(act2, model.conv3.weight, model.conv3.bias, padding=1)
            global_max_out3 = max(global_max_out3, out3.abs().max().item())
            act3 = F.relu(out3)
            
            out4 = F.conv2d(act3, model.conv4.weight, model.conv4.bias, padding=1)
            global_max_out4 = max(global_max_out4, out4.abs().max().item())
            act4 = F.max_pool2d(F.relu(out4), 2)
            
            flat = act4.view(act4.size(0), -1)
            fc_out = F.linear(flat, model.fc.weight, model.fc.bias)
            global_max_fc = max(global_max_fc, fc_out.abs().max().item())

    # 防除零保护算 Scale
    s_in = global_max_in / 127.0 if global_max_in > 0 else 1.0
    s_out1 = global_max_out1 / 127.0 if global_max_out1 > 0 else 1.0
    s_out2 = global_max_out2 / 127.0 if global_max_out2 > 0 else 1.0
    s_out3 = global_max_out3 / 127.0 if global_max_out3 > 0 else 1.0
    s_out4 = global_max_out4 / 127.0 if global_max_out4 > 0 else 1.0
    s_out_fc = global_max_fc / 127.0 if global_max_fc > 0 else 1.0

    print("\n>>> 步骤 3/4: 固化量化参数与权重...")
    w1_q, s_w1 = sym_quantize(model.conv1.weight)
    b1_q = torch.round(model.conv1.bias / (s_in * s_w1)).to(torch.int32)
    mult1, shift1 = calc_multiplier_shift(s_in, s_w1, s_out1)
    
    w2_q, s_w2 = sym_quantize(model.conv2.weight)
    b2_q = torch.round(model.conv2.bias / (s_out1 * s_w2)).to(torch.int32)
    mult2, shift2 = calc_multiplier_shift(s_out1, s_w2, s_out2)
    
    w3_q, s_w3 = sym_quantize(model.conv3.weight)
    b3_q = torch.round(model.conv3.bias / (s_out2 * s_w3)).to(torch.int32)
    mult3, shift3 = calc_multiplier_shift(s_out2, s_w3, s_out3)
    
    w4_q, s_w4 = sym_quantize(model.conv4.weight)
    b4_q = torch.round(model.conv4.bias / (s_out3 * s_w4)).to(torch.int32)
    mult4, shift4 = calc_multiplier_shift(s_out3, s_w4, s_out4)
    
    wfc_q, s_wfc = sym_quantize(model.fc.weight)
    bfc_q = torch.round(model.fc.bias / (s_out4 * s_wfc)).to(torch.int32)
    mult_fc, shift_fc = calc_multiplier_shift(s_out4, s_wfc, s_out_fc)

    print("\n>>> 步骤 4/4: 为靶向目标 (Index=1) 生成 Bit-Accurate 黄金输出...")
    # 严格按照全 INT8 张量流进行前向推演
    target_img_q = torch.clamp(torch.round(target_img_fp32 / s_in), -127, 127).to(torch.int8)

    mac1 = F.conv2d(target_img_q.float(), w1_q.float(), bias=b1_q.float(), padding=1).to(torch.int32)
    q_conv1_out = torch.clamp((mac1 * mult1) >> shift1, -127, 127).to(torch.int8)
    q_act1_out = F.relu(q_conv1_out.float()).to(torch.int8)
    
    mac2 = F.conv2d(q_act1_out.float(), w2_q.float(), bias=b2_q.float(), padding=1).to(torch.int32)
    q_conv2_out = torch.clamp((mac2 * mult2) >> shift2, -127, 127).to(torch.int8)
    q_act2_out = F.relu(q_conv2_out.float()).to(torch.int8)
    q_pool1_out = F.max_pool2d(F.relu(q_conv2_out.float()), 2).to(torch.int8)
    
    mac3 = F.conv2d(q_pool1_out.float(), w3_q.float(), bias=b3_q.float(), padding=1).to(torch.int32)
    q_conv3_out = torch.clamp((mac3 * mult3) >> shift3, -127, 127).to(torch.int8)
    q_act3_out = F.relu(q_conv3_out.float()).to(torch.int8)
    
    mac4 = F.conv2d(q_act3_out.float(), w4_q.float(), bias=b4_q.float(), padding=1).to(torch.int32)
    q_conv4_out = torch.clamp((mac4 * mult4) >> shift4, -127, 127).to(torch.int8)
    q_act4_out = F.relu(q_conv4_out.float()).to(torch.int8)
    q_pool2_out = F.max_pool2d(F.relu(q_conv4_out.float()), 2).to(torch.int8)

    q_flat_in = my_flatten(q_pool2_out)
    mac_fc = F.linear(q_flat_in.float(), wfc_q.float(), bias=bfc_q.float()).to(torch.int32)
    q_fc_out = torch.clamp((mac_fc * mult_fc) >> shift_fc, -127, 127).to(torch.int8)

    predicted_label = torch.argmax(q_fc_out, dim=1).item()
    print(f"🎯 INT8 黄金模型对 Target 的预测结果: {predicted_label} (对比真实标签: {target_label})")

    # ==========================================
    # 5. 导出到大头文件
    # ==========================================
    with open("test_vector.h", "w") as f:
        f.write("#ifndef TEST_VECTOR_H\n#define TEST_VECTOR_H\n\n#include <stdint.h>\n\n")
        f.write("// ================= 全局统一量化参数 =================\n")
        f.write(f"const float GLOBAL_S_IN = {s_in};\n\n")
        
        f.write(f"const int32_t L1_MULT = {mult1};\nconst int32_t L1_SHIFT = {shift1};\n\n")
        f.write(f"const int32_t L2_MULT = {mult2};\nconst int32_t L2_SHIFT = {shift2};\n\n")
        f.write(f"const int32_t L3_MULT = {mult3};\nconst int32_t L3_SHIFT = {shift3};\n\n")
        f.write(f"const int32_t L4_MULT = {mult4};\nconst int32_t L4_SHIFT = {shift4};\n\n")
        f.write(f"const int32_t FC_MULT = {mult_fc};\nconst int32_t FC_SHIFT = {shift_fc};\n\n")
        
        export_c_array(target_img_q, "test_input_img", f, "int8_t")
        
        f.write(f"// ================= Conv1 阶段 =================\n")
        export_c_array(w1_q, "test_conv1_weight", f, "int8_t")
        export_c_array(b1_q, "test_conv1_bias", f, "int32_t")
        f.write(f"// 第一层 Conv + ReLU 后的输出结果 (大小 64x32x32 = 65536)\n")
        export_c_array(q_act1_out, "expected_act1_output", f, "int8_t")
        
        f.write(f"// ================= Conv2 阶段 =================\n")
        export_c_array(w2_q, "test_conv2_weight", f, "int8_t")
        export_c_array(b2_q, "test_conv2_bias", f, "int32_t")
        f.write(f"// 第二层 Conv + ReLU 后的输出结果 (大小 64x32x32 = 65536)\n")
        export_c_array(q_act2_out, "expected_act2_output", f, "int8_t")
        f.write(f"// 第二层 Conv + ReLU + Pool1 后的输出结果 (大小 64x16x16 = 16384)\n")
        export_c_array(q_pool1_out, "expected_pool1_output", f, "int8_t")
        
        f.write(f"// ================= Conv3 阶段 =================\n")
        export_c_array(w3_q, "test_conv3_weight", f, "int8_t")
        export_c_array(b3_q, "test_conv3_bias", f, "int32_t")
        f.write(f"// 第三层 Conv + ReLU 后的输出结果 (大小 64x16x16 = 16384)\n")
        export_c_array(q_act3_out, "expected_act3_output", f, "int8_t")
        
        f.write(f"// ================= Conv4 阶段 =================\n")
        export_c_array(w4_q, "test_conv4_weight", f, "int8_t")
        export_c_array(b4_q, "test_conv4_bias", f, "int32_t")
        f.write(f"// 第四层 Conv + ReLU 后的输出结果 (大小 64x16x16 = 65536)\n")
        export_c_array(q_act4_out, "expected_act4_output", f, "int8_t")
        f.write(f"// 第四层 Conv + ReLU + Pool2 后的输出结果 (大小 64x8x8 = 4096)\n")
        export_c_array(q_pool2_out, "expected_pool2_output", f, "int8_t")
        
        f.write(f"// ================= FC 阶段 =================\n")
        export_c_array(wfc_q, "test_fc_weight", f, "int8_t")
        export_c_array(bfc_q, "test_fc_bias", f, "int32_t")
        f.write(f"// 全连接层最终 10 分类输出\n")
        export_c_array(q_fc_out, "expected_fc_output", f, "int8_t")

        f.write("#endif // TEST_VECTOR_H\n")
    print("静态全局量化的黄金对账单导出完毕！")

if __name__ == "__main__":
    main()

>>> 步骤 1/4: 加载融合版模型与数据集...
Files already downloaded and verified
Files already downloaded and verified
提取黄金测试图片成功 (Index=1)！真实标签: 8

>>> 步骤 2/4: 执行严格的 PTQ 静态校准 (基于训练集)...

>>> 步骤 3/4: 固化量化参数与权重...

>>> 步骤 4/4: 为靶向目标 (Index=1) 生成 Bit-Accurate 黄金输出...
🎯 INT8 黄金模型对 Target 的预测结果: 8 (对比真实标签: 8)
静态全局量化的黄金对账单导出完毕！


## 将test_vector.h原版参数打包为便于PicoRV32+NPU读取的形式
1. 保存在test_vector_2.h中
2. FPGA上板验证的权重参数头文件保存在test_vector_fpga.h
3. 可以直接通过修改TARGET_SYS_COLS来适配不同规模的脉动阵列

In [1]:
import numpy as np
import re

# ==============================================================================
# 1. 提取器 (Extractors)
# ==============================================================================
def parse_c_header_array(content, target_var):
    """提取需要转换的数组并转为 NumPy Array"""
    pattern = rf'const\s+(int8_t|uint8_t|int32_t)\s+({target_var})\[\s*\d*\s*\]\s*=\s*\{{([^}}]+)\}}\s*;'
    match = re.search(pattern, content)
    if not match:
        print(f"[错误] 解析失败：未在文件中找到名为 '{target_var}' 的数组！")
        return None

    dtype_str = match.group(1)
    data_str = match.group(3)
    numbers = [int(x) for x in re.findall(r'-?\d+', data_str)]
    np_dtype = np.int32 if dtype_str == 'int32_t' else np.int8
    
    print(f"[提取-转换] 成功提取 '{target_var}'，类型: {dtype_str}，长度: {len(numbers)}")
    return np.array(numbers, dtype=np_dtype)

def extract_raw_code_block(content, target_var, is_scalar=False):
    """精准抠出不需要转换的 C 语言原始代码块"""
    if is_scalar:
        # 匹配标量: const int32_t L1_MULT = 104;
        pattern = rf'const\s+[a-zA-Z0-9_]+\s+{target_var}\s*=\s*[-0-9xA-Fa-f]+\s*;'
    else:
        # 匹配数组: 使用 re.DOTALL 让 .* 能够跨行匹配花括号内部内容
        pattern = rf'const\s+[a-zA-Z0-9_]+\s+{target_var}\[\s*\d*\s*\]\s*=\s*\{{.*?\}}\s*;'
        
    match = re.search(pattern, content, flags=re.DOTALL)
    if not match:
        print(f"[错误] 未找到原始变量: {target_var}")
        return f"// [Error] {target_var} not found\n"
        
    print(f"[提取-直通] 成功提取原始代码块 '{target_var}'")
    return match.group(0)

# ==============================================================================
# 2. 数据打包引擎 (Packers)
# ==============================================================================
def pack_activation_chw_to_hwc(img_chw):
    C, H, W = img_chw.shape
    pad_c = (4 - (C % 4)) % 4
    if pad_c > 0:
        img_chw = np.pad(img_chw, ((0, pad_c), (0, 0), (0, 0)), mode='constant')
    C_padded = img_chw.shape[0]
    img_hwc = np.transpose(img_chw, (1, 2, 0))

    packed_data = []
    ic_groups = C_padded // 4
    for y in range(H):
        for x in range(W):
            for ig in range(ic_groups):
                base_c = ig * 4
                # 【修复】先强制转为 Python 原生 int，再做掩码
                c0 = int(img_hwc[y, x, base_c + 0]) & 0xFF
                c1 = int(img_hwc[y, x, base_c + 1]) & 0xFF
                c2 = int(img_hwc[y, x, base_c + 2]) & 0xFF
                c3 = int(img_hwc[y, x, base_c + 3]) & 0xFF
                
                # 【修复】拼接后，强制截断为 32-bit 无符号数
                packed_word = ((c3 << 24) | (c2 << 16) | (c1 << 8) | c0) & 0xFFFFFFFF
                packed_data.append(packed_word)
    return packed_data

def pack_chw_activation_to_32bit(chw_array):
    """
    专门用于将 4096 长度的 CHW 一维数组打包为 1024 个 32-bit Word
    :param chw_array: 长度为 4096 的一维 Numpy 数组 (包含 d0 到 d4095)
    :return: 长度为 1024 的 32-bit 无符号整数列表
    """
    # 确保是一维数组
    flat_data = chw_array.flatten()
    
    # 确保长度是 4096 (或者至少能被 4 整除)
    assert len(flat_data) % 4 == 0, "数据总长度必须能被 4 整除！"
    
    packed_data = []
    
    # 每次跳 4 步遍历数据
    for i in range(0, len(flat_data), 4):
        # 提取 d0, d1, d2, d3 并限制为 8-bit 无符号
        d0 = int(flat_data[i + 0]) & 0xFF
        d1 = int(flat_data[i + 1]) & 0xFF
        d2 = int(flat_data[i + 2]) & 0xFF
        d3 = int(flat_data[i + 3]) & 0xFF
        
        # 核心拼接逻辑：d3 移到最高位 (24位)，d0 留在最低位 (0位)
        # 格式严格对应 {d3, d2, d1, d0}
        packed_word = ((d3 << 24) | (d2 << 16) | (d1 << 8) | d0) & 0xFFFFFFFF
        
        packed_data.append(packed_word)
        
    return packed_data

def pack_weights_for_npu(weight_oihw, sys_cols=4):
    OC, IC, KH, KW = weight_oihw.shape
    pad_ic = (4 - (IC % 4)) % 4
    if pad_ic > 0:
        weight_oihw = np.pad(weight_oihw, ((0, 0), (0, pad_ic), (0, 0), (0, 0)), mode='constant')
    IC_padded = weight_oihw.shape[1]
    ic_groups = IC_padded // 4

    packed_data = []
    for oc_start in range(0, OC, sys_cols):
        oc_end = min(oc_start + sys_cols, OC)
        for ig in range(ic_groups):
            for ky in range(KH):
                for kx in range(KW):
                    for oc in range(oc_start, oc_end):
                        # 【修复】先强制转为 Python 原生 int，再做掩码
                        c0 = int(weight_oihw[oc, ig*4 + 0, ky, kx]) & 0xFF
                        c1 = int(weight_oihw[oc, ig*4 + 1, ky, kx]) & 0xFF
                        c2 = int(weight_oihw[oc, ig*4 + 2, ky, kx]) & 0xFF
                        c3 = int(weight_oihw[oc, ig*4 + 3, ky, kx]) & 0xFF
                        
                        # 【修复】拼接后，强制截断为 32-bit 无符号数
                        packed_word = ((c3 << 24) | (c2 << 16) | (c1 << 8) | c0) & 0xFFFFFFFF
                        packed_data.append(packed_word)
    return packed_data

def pack_bias_for_npu(bias_array, sys_cols=4):
    OC = len(bias_array)
    packed_bias = []
    for oc_start in range(0, OC, sys_cols):
        oc_end = min(oc_start + sys_cols, OC)
        for oc in range(oc_start, oc_end):
            val = int(bias_array[oc]) & 0xFFFFFFFF
            packed_bias.append(val)
    return packed_bias

def pack_fc_weights_for_npu(weight_hwc_ready, sys_cols=12):
    """
    完全迎合硬件 FSM 嵌套循环的 HWC 权重打包函数
    硬件循环顺序: Row_Group(0~15) -> Depth(0~63) -> OC_Col(0~11)
    :param weight_hwc_ready: 形状为 (OC, 4096) 的 numpy 数组，且已经按照 HWC 排布展平
    :param sys_cols: NPU 物理列数
    """
    OC, IC_TOTAL = weight_hwc_ready.shape
    
    # OC 对齐到物理列数
    pad_oc = (sys_cols - (OC % sys_cols)) % sys_cols
    if pad_oc > 0:
        weight_padded = np.pad(weight_hwc_ready, ((0, pad_oc), (0, 0)), mode='constant')
    else:
        weight_padded = weight_hwc_ready

    PADDED_OC = weight_padded.shape[0]
    packed_data = []
    
    # 1. 顶层循环：分批处理 OC (一次处理 sys_cols 列)
    for oc_start in range(0, PADDED_OC, sys_cols):
        oc_end = min(oc_start + sys_cols, PADDED_OC)
        
        # 2. 硬件的 Outer Loop: 每次选定 4 行 (Row Group 0~15)
        for row_grp in range(16):
            base_r = row_grp * 4
            
            # 3. 硬件的 Inner Loop: 为这 4 行连续填满 64 个深度 (Depth 0~63)
            for depth in range(64):
                
                # 4. 硬件的最内层 Loop: 遍历实际列 (OC 0~11)
                for oc in range(oc_start, oc_end):
                    
                    # 【核心映射逻辑】
                    # 在 HWC 中，每一拍吃 64 个 C，所以 depth(0~63) 相当于空间(Spatial)坐标。
                    # base_r(0~63) 相当于通道(Channel)坐标。
                    # 1D 索引 = spatial * 64 + channel = depth * 64 + base_r
                    idx0 = depth * 64 + base_r + 0
                    idx1 = depth * 64 + base_r + 1
                    idx2 = depth * 64 + base_r + 2
                    idx3 = depth * 64 + base_r + 3
                    
                    c0 = int(weight_padded[oc, idx0]) & 0xFF
                    c1 = int(weight_padded[oc, idx1]) & 0xFF
                    c2 = int(weight_padded[oc, idx2]) & 0xFF
                    c3 = int(weight_padded[oc, idx3]) & 0xFF
                    
                    # 小端序拼接：c3 在最高位，c0 在最低位
                    packed_word = ((c3 << 24) | (c2 << 16) | (c1 << 8) | c0) & 0xFFFFFFFF
                    packed_data.append(packed_word)
                    
    return packed_data
# ==============================================================================
# 主执行入口：流水线调度器
# ==============================================================================
if __name__ == "__main__":
    SOURCE_FILE = "test_vector.h"
    OUTPUT_FILE = "test_vector_2.h"
    TARGET_SYS_COLS = 8

    # 读取原始文件全部内容
    try:
        with open(SOURCE_FILE, 'r', encoding='utf-8') as f:
            content = f.read()
    except FileNotFoundError:
        print(f"[致命错误] 找不到源文件 {SOURCE_FILE}")
        exit(1)

    # 初始化输出文件缓存区
    out_lines = [
        "#ifndef TEST_VECTOR_H",
        "#define TEST_VECTOR_H",
        "",
        "#include <stdint.h>",
        ""
    ]

    # 工具函数：生成转换后的 C 数组字符串并追加
    def append_converted_array(var_name, packed_data):
        out_lines.append(f"// ==========================================================")
        out_lines.append(f"// NPU 格式重排数据 (Aligned 32-bit): {var_name}")
        out_lines.append(f"// ==========================================================")
        # 强制四字节对齐 __attribute__((aligned(4)))
        out_lines.append(f"const uint32_t {var_name}[{len(packed_data)}] __attribute__((aligned(4))) = {{")
        for i in range(0, len(packed_data), 8):
            chunk = packed_data[i:i+8]
            chunk_str = ", ".join([f"0x{x:08x}" for x in chunk])
            out_lines.append(f"    {chunk_str},")
        out_lines.append("};\n")

    # ---------------------------------------------------------
    # 步骤 A：处理需要被转换的数据 (Activations, Weights, Biases)
    # ---------------------------------------------------------
    # 1. 激活值 (此处假设第一层输入名为 test_input_img)
    act_var = "test_input_img" 
    raw_act = parse_c_header_array(content, target_var=act_var)
    if raw_act is not None:
        img_chw = raw_act.reshape(3, 32, 32)
        packed_act = pack_activation_chw_to_hwc(img_chw)
        append_converted_array(act_var, packed_act)
    
    # 1. 激活值2
    act_var2 = "expected_act1_output" 
    raw_act2 = parse_c_header_array(content, target_var=act_var2)
    if raw_act2 is not None:
        img_chw2 = raw_act2.reshape(64, 32, 32)
        packed_act2 = pack_activation_chw_to_hwc(img_chw2)
        append_converted_array(act_var2, packed_act2)
    
    # 1. 激活值2
    act_var2 = "expected_act2_output" 
    raw_act2 = parse_c_header_array(content, target_var=act_var2)
    if raw_act2 is not None:
        img_chw2 = raw_act2.reshape(64, 32, 32)
        packed_act2 = pack_activation_chw_to_hwc(img_chw2)
        append_converted_array(act_var2, packed_act2)
    
    # 1. 激活值3
    act_var2 = "expected_pool1_output" 
    raw_act2 = parse_c_header_array(content, target_var=act_var2)
    if raw_act2 is not None:
        img_chw2 = raw_act2.reshape(64, 16, 16)
        packed_act2 = pack_activation_chw_to_hwc(img_chw2)
        append_converted_array(act_var2, packed_act2)
    
    # 1. 激活值4
    act_var2 = "expected_act3_output" 
    raw_act2 = parse_c_header_array(content, target_var=act_var2)
    if raw_act2 is not None:
        img_chw2 = raw_act2.reshape(64, 16, 16)
        packed_act2 = pack_activation_chw_to_hwc(img_chw2)
        append_converted_array(act_var2, packed_act2)

    # 1. 激活值2
    act_var2 = "expected_act4_output" 
    raw_act2 = parse_c_header_array(content, target_var=act_var2)
    if raw_act2 is not None:
        img_chw2 = raw_act2.reshape(64, 16, 16)
        packed_act2 = pack_activation_chw_to_hwc(img_chw2)
        append_converted_array(act_var2, packed_act2)
    
    # 1. 激活值5
    act_var2 = "expected_pool2_output" 
    raw_act2 = parse_c_header_array(content, target_var=act_var2)
    if raw_act2 is not None:
        img_chw2 = raw_act2.reshape(64, 8, 8)
        packed_act2 = pack_activation_chw_to_hwc(img_chw2)
        append_converted_array(act_var2, packed_act2)

    # 2. 卷积层 1
    raw_w1 = parse_c_header_array(content, "test_conv1_weight")
    if raw_w1 is not None:
        packed_w1 = pack_weights_for_npu(raw_w1.reshape(64, 3, 3, 3), sys_cols=TARGET_SYS_COLS)
        append_converted_array("test_conv1_weight", packed_w1)

    raw_b1 = parse_c_header_array(content, "test_conv1_bias")
    if raw_b1 is not None:
        packed_b1 = pack_bias_for_npu(raw_b1, sys_cols=TARGET_SYS_COLS)
        append_converted_array("test_conv1_bias", packed_b1)

    # 3. 卷积层 2
    raw_w2 = parse_c_header_array(content, "test_conv2_weight")
    if raw_w2 is not None:
        # Conv2: OC=32, IC=16, KH=3, KW=3 (总共 4608 个元素)
        packed_w2 = pack_weights_for_npu(raw_w2.reshape(64, 64, 3, 3), sys_cols=TARGET_SYS_COLS)
        append_converted_array("test_conv2_weight", packed_w2)

    raw_b2 = parse_c_header_array(content, "test_conv2_bias")
    if raw_b2 is not None:
        packed_b2 = pack_bias_for_npu(raw_b2, sys_cols=TARGET_SYS_COLS)
        append_converted_array("test_conv2_bias", packed_b2)

    # 4. 卷积层 3
    raw_w3 = parse_c_header_array(content, "test_conv3_weight")
    if raw_w3 is not None:
        packed_w3 = pack_weights_for_npu(raw_w3.reshape(64, 64, 3, 3), sys_cols=TARGET_SYS_COLS)
        append_converted_array("test_conv3_weight", packed_w3)

    raw_b3 = parse_c_header_array(content, "test_conv3_bias")
    if raw_b3 is not None:
        packed_b3 = pack_bias_for_npu(raw_b3, sys_cols=TARGET_SYS_COLS)
        append_converted_array("test_conv3_bias", packed_b3)

    # 5. 卷积层 4
    raw_w4 = parse_c_header_array(content, "test_conv4_weight")
    if raw_w4 is not None:
        # Conv2: OC=32, IC=16, KH=3, KW=3 (总共 4608 个元素)
        packed_w4 = pack_weights_for_npu(raw_w4.reshape(64, 64, 3, 3), sys_cols=TARGET_SYS_COLS)
        append_converted_array("test_conv4_weight", packed_w4)

    raw_b4 = parse_c_header_array(content, "test_conv4_bias")
    if raw_b4 is not None:
        packed_b4 = pack_bias_for_npu(raw_b4, sys_cols=TARGET_SYS_COLS)
        append_converted_array("test_conv4_bias", packed_b4)

    raw_wfc = parse_c_header_array(content, "test_fc_weight")
    if raw_wfc is not None:
        # 2. 还原回 CHW 的三维空间视角 (64 通道，8x8 宽高)
        # NumPy 中直接使用 reshape 即可
        fc_weight_chw = raw_wfc.reshape(10, 64, 8, 8)

        # 3. 数据布局转换：将 CHW 排布转换为 HWC
        # 【修正】NumPy 的 transpose 对应 PyTorch 的 permute
        # 【修正】NumPy 的 copy() 对应 PyTorch 的 contiguous()，确保内存重新连续排列
        fc_weight_hwc = fc_weight_chw.transpose(0, 2, 3, 1).copy()

        # 4. 再次展平为 (10, 4096)，此时矩阵已匹配 NPU 的 HWC 输出格式
        # NumPy 中仍然使用 reshape 替代 PyTorch 的 view
        fc_weight_ready = fc_weight_hwc.reshape(10, 4096)
        
        # 5. 调用打包函数
        packed_b4 = pack_fc_weights_for_npu(fc_weight_ready, sys_cols=8)
        append_converted_array("test_fc_weight", packed_b4)

    raw_wfc = parse_c_header_array(content, "test_fc_bias")
    if raw_wfc is not None:
        packed_b4 = pack_bias_for_npu(raw_wfc, sys_cols=8)
        append_converted_array("test_fc_bias", packed_b4)
    # ---------------------------------------------------------
    # 步骤 B：处理原样直通的数据 (Raw Data)
    # ---------------------------------------------------------
    out_lines.append(f"// ==========================================================")
    out_lines.append(f"// 以下为原样直通数据 (未重排的原始格式)")
    out_lines.append(f"// ==========================================================\n")

    raw_arrays_to_copy = [
        # "expected_act1_output",
        # "expected_pool2_output",
        # "test_fc_weight",
        # "test_fc_bias",
        "expected_fc_output"
    ]
    for arr_name in raw_arrays_to_copy:
        raw_code = extract_raw_code_block(content, arr_name, is_scalar=False)
        out_lines.append(raw_code + "\n")

    raw_scalars_to_copy = [
        "L1_MULT", "L1_SHIFT",
        "L2_MULT", "L2_SHIFT",
        "L3_MULT", "L3_SHIFT",
        "L4_MULT", "L4_SHIFT",
        "FC_MULT", "FC_SHIFT"
    ]
    for scalar_name in raw_scalars_to_copy:
        raw_code = extract_raw_code_block(content, scalar_name, is_scalar=True)
        out_lines.append(raw_code)
    
    out_lines.append("\n#endif // TEST_VECTOR_H\n")

    # ---------------------------------------------------------
    # 步骤 C：一次性刷入文件
    # ---------------------------------------------------------
    with open(OUTPUT_FILE, 'w', encoding='utf-8') as f:
        f.write("\n".join(out_lines))
    print(f"\n[完成] 成功构建终极头文件并写入：{OUTPUT_FILE}")

[提取-转换] 成功提取 'test_input_img'，类型: int8_t，长度: 3072
[提取-转换] 成功提取 'expected_act1_output'，类型: int8_t，长度: 65536
[提取-转换] 成功提取 'expected_act2_output'，类型: int8_t，长度: 65536
[提取-转换] 成功提取 'expected_pool1_output'，类型: int8_t，长度: 16384
[提取-转换] 成功提取 'expected_act3_output'，类型: int8_t，长度: 16384
[提取-转换] 成功提取 'expected_act4_output'，类型: int8_t，长度: 16384
[提取-转换] 成功提取 'expected_pool2_output'，类型: int8_t，长度: 4096
[提取-转换] 成功提取 'test_conv1_weight'，类型: int8_t，长度: 1728
[提取-转换] 成功提取 'test_conv1_bias'，类型: int32_t，长度: 64
[提取-转换] 成功提取 'test_conv2_weight'，类型: int8_t，长度: 36864
[提取-转换] 成功提取 'test_conv2_bias'，类型: int32_t，长度: 64
[提取-转换] 成功提取 'test_conv3_weight'，类型: int8_t，长度: 36864
[提取-转换] 成功提取 'test_conv3_bias'，类型: int32_t，长度: 64
[提取-转换] 成功提取 'test_conv4_weight'，类型: int8_t，长度: 36864
[提取-转换] 成功提取 'test_conv4_bias'，类型: int32_t，长度: 64
[提取-转换] 成功提取 'test_fc_weight'，类型: int8_t，长度: 40960
[提取-转换] 成功提取 'test_fc_bias'，类型: int32_t，长度: 10
[提取-直通] 成功提取原始代码块 'expected_fc_output'
[提取-直通] 成功提取原始代码块 'L1_MULT'
[提取-直通] 成功提取原始代码块 'L1_SHIFT